# Day 1 — When to Fine-Tune (and When Not To)

---

Before you spend a weekend fine-tuning, answer honestly: **do you actually need to?** For 70% of AI-engineering tasks, the answer is *no* — a better prompt or a RAG pipeline will do the job faster, cheaper, and with less risk.

Today:

1. The **decision tree** — prompt engineering → RAG → fine-tuning
2. What fine-tuning actually changes (and doesn't)
3. Preparing a good **fine-tuning dataset** — the single biggest lever


## 1. The three-tier decision

| Approach | Changes | When it wins |
|---|---|---|
| **Prompt engineering** | Nothing about the model | Behavior can be described in words |
| **RAG** | Adds knowledge at query time | Knowledge changes often, or is private |
| **Fine-tuning** | Model weights themselves | Style / format / niche skill the model can't do reliably even with a great prompt |

**Rule of thumb:** exhaust prompt engineering, then try RAG. Only reach for fine-tuning when both fail — or when latency / cost demands a smaller model that behaves like a bigger one.

**Concrete examples:**

- "Answer as our brand voice" → **fine-tune** (style is hard to prompt reliably)
- "Answer using our internal docs" → **RAG** (knowledge, not style)
- "Return valid JSON" → **prompt** (models are already good at this)
- "Extract entities from Sanskrit legal texts" → **fine-tune** (rare domain, base model is weak)


## 2. What fine-tuning does & doesn't do

**Does:**
- Teach a **style or format** (bullet points, brand tone, code style)
- Teach a **narrow skill** (legal case classification, medical coding, SQL from question)
- Make a **small model** behave more like a big one on your task
- Reduce **prompt length** — you don't have to spell out formatting rules every time

**Does NOT:**
- Add new **factual knowledge** reliably (fine-tuning is bad at memorizing facts — use RAG)
- Fix **reasoning** deficits without a lot of data
- Guarantee the model won't hallucinate (the base model still owns most weights)


## 3. Instruction-tuning format — the standard shape

Modern fine-tunes almost all use one of two formats:

**a) Chat format** (ChatML / Llama-3 template)
```json
{"messages": [
  {"role": "system", "content": "..."},
  {"role": "user",   "content": "..."},
  {"role": "assistant", "content": "..."}
]}
```

**b) Instruction format** (Alpaca-style)
```json
{"instruction": "...", "input": "...", "output": "..."}
```

Both work. **Chat format is what most 2026 models expect**, so we'll use it.


## 4. Dataset size — how much is enough?

Fewer examples than you probably think.

| Goal | Rough minimum | Sweet spot |
|---|---|---|
| Style / format learning | ~50 | 200–500 |
| Narrow classification task | ~200 | 500–2000 |
| Complex domain skill (legal / medical) | ~1000 | 3000–10000 |
| New language / massive style shift | 10k+ | 50k+ |

**Quality >> quantity.** 200 hand-crafted examples beat 5000 scraped ones. Almost always.


## 5. Building a good dataset — checklist

- **Every example should reflect the behavior you want.** No junk, no half-hearted labels.
- **Diverse inputs.** If all your examples are one-liners, the model won't learn multi-paragraph inputs.
- **Consistent output format.** If some outputs use bullets and others use paragraphs, the model learns confusion.
- **A held-out eval set (10–20% of the data).** You need this to know if training actually helped.
- **Include failure modes.** If the base model sometimes hallucinates numbers, include examples that show the correct restrained behavior.


## 6. A tiny sample dataset

Let's shape a 6-example fine-tuning dataset for a "customer support ticket categorizer."


In [ ]:
import json

data = [
    {"messages": [
        {"role": "system",    "content": "You are a support triage bot. Classify each ticket into exactly one of: billing, technical, feature-request, other. Respond with only the label."},
        {"role": "user",      "content": "My card was charged twice for last month."},
        {"role": "assistant", "content": "billing"},
    ]},
    {"messages": [
        {"role": "system",    "content": "You are a support triage bot. Classify each ticket into exactly one of: billing, technical, feature-request, other. Respond with only the label."},
        {"role": "user",      "content": "The dashboard is not loading in Chrome."},
        {"role": "assistant", "content": "technical"},
    ]},
    {"messages": [
        {"role": "system",    "content": "You are a support triage bot. Classify each ticket into exactly one of: billing, technical, feature-request, other. Respond with only the label."},
        {"role": "user",      "content": "It would be great if you could add dark mode."},
        {"role": "assistant", "content": "feature-request"},
    ]},
    {"messages": [
        {"role": "system",    "content": "You are a support triage bot. Classify each ticket into exactly one of: billing, technical, feature-request, other. Respond with only the label."},
        {"role": "user",      "content": "Why is my monthly bill higher this month?"},
        {"role": "assistant", "content": "billing"},
    ]},
    {"messages": [
        {"role": "system",    "content": "You are a support triage bot. Classify each ticket into exactly one of: billing, technical, feature-request, other. Respond with only the label."},
        {"role": "user",      "content": "The API returns 500 errors when I POST large files."},
        {"role": "assistant", "content": "technical"},
    ]},
    {"messages": [
        {"role": "system",    "content": "You are a support triage bot. Classify each ticket into exactly one of: billing, technical, feature-request, other. Respond with only the label."},
        {"role": "user",      "content": "Can you tell me what time your office is open?"},
        {"role": "assistant", "content": "other"},
    ]},
]

# Save as JSONL - the standard fine-tuning format
with open("triage.jsonl", "w") as f:
    for ex in data:
        f.write(json.dumps(ex) + "\n")

print(f"Saved {len(data)} examples to triage.jsonl")


**JSONL** (one JSON object per line) is the standard file format for Hugging Face `datasets`, OpenAI fine-tuning, and Together AI fine-tuning. If you export in this shape, you can move providers easily.


## 7. Train / eval split


In [ ]:
import random

random.seed(42)
random.shuffle(data)
split = int(len(data) * 0.8)
train, eval_ = data[:split], data[split:]

with open("triage_train.jsonl", "w") as f:
    for ex in train: f.write(json.dumps(ex) + "\n")
with open("triage_eval.jsonl", "w") as f:
    for ex in eval_: f.write(json.dumps(ex) + "\n")

print(f"train: {len(train)}   eval: {len(eval_)}")


**Never train on your eval set.** It's the single most common junior-engineer fine-tuning mistake. Split *once*, then don't touch the eval file until measurement time.


## Recap

- Fine-tune only after prompt engineering and RAG both fall short.
- Great for **style, format, narrow skills**. Bad for **new facts** (that's RAG).
- **Chat-format JSONL** is the modern standard.
- **200–500 hand-crafted examples** is enough for most style/classification tasks.
- Always keep a **held-out eval set**.
- **Next class:** LoRA / QLoRA — the cheap way to actually train the model.
